# Pretrained OmniLearn / PET Practice Notebook

This notebook is a project-start rehearsal for using a pretrained OmniLearn / PET-style jet foundation model in this repository.

It is intentionally conservative:

- it does not launch full training by default;
- it does not download JetClass or checkpoint artifacts automatically;
- it checks the local Python environment, optional OmniLearn reference checkout, JetClass-like HDF5 files, checkpoint files, and command templates;
- it records the evidence needed before the project moves from preparation into formal E0/E0.5/E1 experiments.

Project framing: the formal study will fine-tune a pretrained OmniLearn / PET-style backbone, or a later explicitly selected OmniLearned-compatible checkpoint, on JetClass binary top-vs-QCD tagging under nested feature configurations A-D. This practice notebook is only for environment familiarity and reproducibility rehearsal.

## Practice Goals

By the end of this notebook, you should be able to answer:

1. Is the local Python environment close enough to run the OmniLearn reference workflow?
2. Is an OmniLearn reference repository checkout available and recognizable?
3. Are local JetClass or JetClass-like HDF5 files present, and do they expose `data`, `jet`, and `pid` datasets?
4. Can candidate A-D feature configurations be mapped from actual particle-level columns?
5. Are pretrained checkpoint files present, and can their hashes be recorded?
6. What exact commands should be run later for preprocessing, fine-tuning, and evaluation?

The notebook follows the official OmniLearn repository interface where possible: `data` has shape `(N, P, F)`, `jet` has shape `(N, J)`, and `pid` stores classification labels.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import json
import platform
import re
import subprocess
import sys
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np

try:
    import h5py
except ImportError:
    h5py = None

try:
    from packaging.requirements import InvalidRequirement, Requirement
except ImportError:
    InvalidRequirement = ValueError
    Requirement = None


def resolve_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook, terminal, or exported script context."""
    override = globals().get("PROJECT_ROOT_OVERRIDE")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        anchor = (start or Path.cwd()).resolve()
        candidates = [anchor, *anchor.parents]

    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "src" / "notebooks").exists():
            return candidate

    searched = ", ".join(str(candidate) for candidate in candidates[:6])
    raise RuntimeError(
        "Could not resolve project root. Set PROJECT_ROOT_OVERRIDE to the particleML repository path. "
        f"Searched: {searched}"
    )


def safe_git_value(args: Sequence[str], cwd: Path) -> str | None:
    try:
        return subprocess.check_output(["git", *args], cwd=cwd, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "jetclass"
NOTEBOOK_DIR = PROJECT_ROOT / "src" / "notebooks"
REFERENCE_DIR = PROJECT_ROOT / "external" / "OmniLearn"
CHECKPOINT_CANDIDATE_DIRS = [
    REFERENCE_DIR / "checkpoints",
    DATA_DIR / "checkpoints",
    PROJECT_ROOT / "checkpoints",
]
FEATURE_LADDER_PATH = PROJECT_ROOT / "src" / "config" / "jetclass_feature_ladder.json"
MASK_FEATURE_INDEX = 2

OMNILEARN_REPO_URL = "https://github.com/ViniciusMikuni/OmniLearn"
OMNILEARN_PAPER = "https://arxiv.org/abs/2404.16091"
JETCLASS_ZENODO = "https://zenodo.org/records/6619768"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Reference OmniLearn checkout:", REFERENCE_DIR)
print("Notebook directory:", NOTEBOOK_DIR)

## Optional Setup

Run this cell only if the environment inventory below reports missing practice packages. In local development, activate the intended virtual environment before running it. In Google Colab, run it once near the start of the session and restart the kernel if imports still fail.

This installs only the minimum packages needed for the practice notebook's HDF5 and tabular checks; it is not the full OmniLearn training environment.


In [ ]:
# Optional: install minimum packages needed for this practice notebook.
# Keep this commented until the environment check reports missing packages.
# %pip install -q h5py pandas scikit-learn


## 1. Environment Inventory

This cell records the runtime environment. Treat missing packages as setup work, not as a research result.

The official OmniLearn repository currently lists TensorFlow/Keras, h5py, NumPy, pandas, scikit-learn, SciPy, PyTorch, PyTorch Geometric, Horovod, EnergyFlow, uproot, vector, and related HEP/ML packages. The official recommendation is to use the provided Docker container when reproducing the reference workflow.

In [ ]:
REQUIRED_FOR_PRACTICE = [
    "numpy",
    "h5py",
    "pandas",
    "sklearn",
    "scipy",
    "matplotlib",
]

REFERENCE_STACK = [
    "tensorflow",
    "keras",
    "torch",
    "torch_geometric",
    "torch_cluster",
    "awkward",
    "coffea",
    "EnergyFlow",
    "uproot",
    "vector",
    "horovod",
]

PACKAGE_NAME_MAP = {
    "sklearn": "scikit-learn",
    "torch_geometric": "torch-geometric",
    "torch_cluster": "torch-cluster",
}


def package_version(import_name: str) -> str:
    dist_name = PACKAGE_NAME_MAP.get(import_name, import_name)
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return "MISSING"


def check_reference_requirements(requirements_path: Path) -> dict[str, object]:
    report: dict[str, object] = {
        "path": str(requirements_path),
        "checked": False,
        "compatible": False,
        "missing": [],
        "incompatible": [],
        "skipped": [],
        "error": None,
    }
    if not requirements_path.exists():
        report["error"] = "requirements.txt not found"
        return report
    if Requirement is None:
        report["error"] = "packaging is not installed"
        return report
    report["checked"] = True
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith(("-", "git+", "http://", "https://")):
            report["skipped"].append(line)
            continue
        try:
            requirement = Requirement(line)
        except InvalidRequirement:
            report["skipped"].append(line)
            continue
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            installed_version = importlib_metadata.version(requirement.name)
        except importlib_metadata.PackageNotFoundError:
            report["missing"].append(requirement.name)
            continue
        if requirement.specifier and installed_version not in requirement.specifier:
            report["incompatible"].append({
                "package": requirement.name,
                "installed": installed_version,
                "required": str(requirement.specifier),
            })
    report["missing"].sort()
    report["compatible"] = not report["missing"] and not report["incompatible"] and not report["error"]
    return report


def detect_gpu_hardware() -> dict[str, object]:
    report: dict[str, object] = {"gpu_names": [], "gpu_vram_mb": [], "cuda_version": None}
    try:
        query = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True, timeout=10,
        )
        rows = [line.strip() for line in query.stdout.splitlines() if line.strip()]
        report["gpu_names"] = [row.rsplit(",", 1)[0].strip() for row in rows]
        report["gpu_vram_mb"] = [int(row.rsplit(",", 1)[1].strip()) for row in rows]
        summary = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True, timeout=10)
        cuda_match = re.search(r"CUDA Version:\s*([0-9.]+)", summary.stdout)
        report["cuda_version"] = cuda_match.group(1) if cuda_match else None
    except (FileNotFoundError, subprocess.SubprocessError, ValueError):
        pass
    return report


package_versions = {name: package_version(name) for name in REQUIRED_FOR_PRACTICE + REFERENCE_STACK}
missing_practice = [name for name in REQUIRED_FOR_PRACTICE if package_versions[name] == "MISSING"]
missing_reference_stack = [name for name in REFERENCE_STACK if package_versions[name] == "MISSING"]
env_report = {
    "python": sys.version.replace("\n", " "),
    "executable_name": Path(sys.executable).name,
    "platform": platform.platform(),
    "packages": package_versions,
    "practice_ready": not missing_practice,
    "reference_stack_ready": not missing_reference_stack,
    "missing_reference_stack": missing_reference_stack,
    **detect_gpu_hardware(),
}

print(json.dumps(env_report, indent=2))

if missing_practice:
    print("\nMissing practice packages:", missing_practice)
else:
    print("\nMinimum practice package set is available.")

if missing_reference_stack:
    print("Reference workflow is NOT ready; missing packages:", missing_reference_stack)
else:
    print("Reference package stack is present. Verify exact versions against the checked-out requirements.txt before training.")

## 2. Optional OmniLearn Reference Checkout

The notebook expects the official OmniLearn reference repository at `external/OmniLearn` by default. Keeping it under `external/` avoids mixing third-party reference code with this project's future implementation.

Run the clone command manually in a terminal when needed:

```bash
git clone https://github.com/ViniciusMikuni/OmniLearn.git external/OmniLearn
```

The `checkpoints/` directory is treated as optional because pretrained weights may be downloaded or copied separately from a clean source checkout. If the clone fails because of network instability, retry later or download an archive from GitHub. The rest of this notebook can still run without the checkout.

In [ ]:
REQUIRED_OMNILEARN_PATHS = [
    "README.md",
    "requirements.txt",
    "preprocessing",
    "scripts",
]
OPTIONAL_OMNILEARN_PATHS = [
    "checkpoints",
]


def describe_reference_checkout(path: Path) -> dict[str, object]:
    exists = path.exists()
    report: dict[str, object] = {
        "path": str(path),
        "exists": exists,
        "required_paths": {},
        "optional_paths": {},
        "recognizable": False,
        "reference_ready": False,
        "requirements_compatibility": None,
        "is_git_checkout": False,
        "git_commit": None,
        "git_describe": None,
        "paper_url": OMNILEARN_PAPER,
    }
    if exists:
        for rel in REQUIRED_OMNILEARN_PATHS:
            report["required_paths"][rel] = (path / rel).exists()
        for rel in OPTIONAL_OMNILEARN_PATHS:
            report["optional_paths"][rel] = (path / rel).exists()
        report["recognizable"] = all(report["required_paths"].values())
        report["requirements_compatibility"] = check_reference_requirements(path / "requirements.txt")
        report["is_git_checkout"] = (path / ".git").exists()
        if report["is_git_checkout"]:
            report["git_commit"] = safe_git_value(["rev-parse", "HEAD"], cwd=path)
            report["git_describe"] = safe_git_value(["describe", "--tags", "--always"], cwd=path)
        reference_stack_ready = globals().get("env_report", {}).get("reference_stack_ready", False)
        report["reference_ready"] = bool(
            report["recognizable"]
            and reference_stack_ready
            and report["requirements_compatibility"]["compatible"]
        )
    return report


reference_report = describe_reference_checkout(REFERENCE_DIR)
print(json.dumps(reference_report, indent=2))

if not reference_report["recognizable"]:
    if reference_report["exists"]:
        missing_paths = [name for name, present in reference_report["required_paths"].items() if not present]
        print("\nReference directory exists but is incomplete; missing required paths:", missing_paths)
    else:
        print("\nReference checkout not found. Suggested command:")
        print(f"git clone {OMNILEARN_REPO_URL}.git {REFERENCE_DIR}")

## 3. Local HDF5 Discovery

This section inspects local HDF5 files under `data/jetclass`. It accepts `.h5` and `.hdf5` files.

For the official OmniLearn-compatible format, each file should contain:

- `data`: particle point clouds with shape `(N, P, F)`;
- `jet`: jet-level features with shape `(N, J)`;
- `pid`: classification labels, often one-hot encoded for classifier tasks.

This is a schema check only. It does not validate physics correctness, official splits, event identity leakage, or feature semantics. The source of truth for JetClass downloads is the Zenodo record: `https://zenodo.org/records/6619768`.

In [ ]:
HDF5_SUFFIXES = {".h5", ".hdf5"}


def discover_hdf5_files(base_dir: Path, excluded_dirs: Iterable[Path] = ()) -> list[Path]:
    if not base_dir.exists():
        return []
    excluded_roots = [path.expanduser().resolve() for path in excluded_dirs]
    return sorted(
        path for path in base_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in HDF5_SUFFIXES
        and not any(path.resolve().is_relative_to(root) for root in excluded_roots)
    )


def dataset_summary(obj) -> dict[str, object]:
    return {
        "shape": tuple(int(x) for x in obj.shape),
        "dtype": str(obj.dtype),
        "ndim": int(obj.ndim),
    }


def summarize_hdf5_file(path: Path, max_keys: int = 20) -> dict[str, object]:
    if h5py is None:
        return {"path": str(path), "error": "h5py is not installed"}
    summary: dict[str, object] = {"path": str(path), "datasets": {}, "groups": []}
    try:
        with h5py.File(path, "r") as handle:
            all_keys = list(handle.keys())
            keys = all_keys[:max_keys]
            summary["top_level_keys"] = keys
            summary["total_top_level_keys"] = len(all_keys)
            summary["inspected_top_level_keys"] = len(keys)
            summary["truncated"] = len(all_keys) > max_keys
            for key in keys:
                obj = handle[key]
                if hasattr(obj, "shape"):
                    summary["datasets"][key] = dataset_summary(obj)
                else:
                    summary["groups"].append(key)
    except Exception as exc:
        summary["error"] = f"{type(exc).__name__}: {exc}"
    return summary


hdf5_files = discover_hdf5_files(DATA_DIR, excluded_dirs=CHECKPOINT_CANDIDATE_DIRS)
print(f"Found {len(hdf5_files)} HDF5 file(s) under {DATA_DIR}")
for path in hdf5_files[:10]:
    print("-", path.relative_to(PROJECT_ROOT))

if hdf5_files:
    first_summary = summarize_hdf5_file(hdf5_files[0])
    print("\nFirst file summary:")
    print(json.dumps(first_summary, indent=2))
else:
    print("\nNo local HDF5 files found. Put a small JetClass dev slice under data/jetclass before running schema checks.")

## 4. OmniLearn Input-Contract Check

This cell checks whether a file looks compatible with the official OmniLearn custom dataset contract. Passing this check only means the basic keys and tensor ranks look plausible.

In [ ]:
@dataclass(frozen=True)
class InputContractResult:
    path: str
    has_data: bool
    has_jet: bool
    has_pid: bool
    data_shape: tuple[int, ...] | None
    jet_shape: tuple[int, ...] | None
    pid_shape: tuple[int, ...] | None
    issues: tuple[str, ...]
    error: str | None = None


def check_omnilearn_input_contract(path: Path) -> InputContractResult:
    issues: list[str] = []
    if h5py is None:
        return InputContractResult(str(path), False, False, False, None, None, None, ("h5py is not installed",))

    try:
        with h5py.File(path, "r") as handle:
            has_data = "data" in handle
            has_jet = "jet" in handle
            has_pid = "pid" in handle

            data_shape = tuple(handle["data"].shape) if has_data else None
            jet_shape = tuple(handle["jet"].shape) if has_jet else None
            pid_shape = tuple(handle["pid"].shape) if has_pid else None

            if not has_data:
                issues.append("missing 'data' dataset")
            elif len(data_shape) != 3:
                issues.append(f"'data' should have rank 3 (N, P, F), got {data_shape}")

            if not has_jet:
                issues.append("missing 'jet' dataset")
            elif len(jet_shape) != 2:
                issues.append(f"'jet' should have rank 2 (N, J), got {jet_shape}")

            if not has_pid:
                issues.append("missing 'pid' dataset")
            elif len(pid_shape) not in {1, 2}:
                issues.append(f"'pid' should have rank 1 or 2, got {pid_shape}")

            n_values = [shape[0] for shape in [data_shape, jet_shape, pid_shape] if shape]
            if not n_values:
                issues.append("no comparable leading dimensions because no required datasets were found")
            elif any(n == 0 for n in n_values):
                issues.append(f"one or more required datasets have zero leading dimension: {n_values}")
            elif len(set(n_values)) > 1:
                issues.append(f"leading dimensions disagree: {n_values}")
    except Exception as exc:
        return InputContractResult(
            path=str(path),
            has_data=False,
            has_jet=False,
            has_pid=False,
            data_shape=None,
            jet_shape=None,
            pid_shape=None,
            issues=("could not open or parse HDF5 file",),
            error=f"{type(exc).__name__}: {exc}",
        )

    return InputContractResult(
        path=str(path),
        has_data=has_data,
        has_jet=has_jet,
        has_pid=has_pid,
        data_shape=data_shape,
        jet_shape=jet_shape,
        pid_shape=pid_shape,
        issues=tuple(issues),
    )


contract_results = [check_omnilearn_input_contract(path) for path in hdf5_files[:10]]
for result in contract_results:
    print(json.dumps(result.__dict__, indent=2))
    print()

## 5. Inspect One Batch-Like Slice

Use a tiny slice to inspect masks, finite values, and label layout. This helps catch environment and data-format problems before any model code is involved.

In [ ]:
def finite_fraction(array: np.ndarray) -> float:
    if array.size == 0:
        return float("nan")
    return float(np.isfinite(array).mean())


def preview_nested(value: object, max_items: int = 3) -> object:
    if isinstance(value, list):
        return [preview_nested(item, max_items=max_items) for item in value[:max_items]]
    return value


def derive_particle_mask(data: np.ndarray, mask_feature_index: int = MASK_FEATURE_INDEX) -> np.ndarray:
    data = np.asarray(data)
    if data.ndim != 3:
        raise ValueError(f"Expected particle data with rank 3 (N, P, F), got {data.shape}")
    if mask_feature_index < 0 or mask_feature_index >= data.shape[-1]:
        raise IndexError(f"Mask feature index {mask_feature_index} is outside [0, {data.shape[-1]})")
    mask_values = data[..., mask_feature_index]
    if not np.isfinite(mask_values).all():
        raise ValueError("Mask source column contains NaN or Inf")
    return mask_values != 0


def inspect_tiny_slice(path: Path, n: int = 8) -> dict[str, object]:
    if h5py is None:
        return {"error": "h5py is not installed"}
    try:
        with h5py.File(path, "r") as handle:
            report: dict[str, object] = {"path": str(path), "n": n}
            if "data" in handle:
                data = np.asarray(handle["data"][:n])
                abs_sum = np.abs(data).sum(axis=-1)
                particle_mask = derive_particle_mask(data)
                report["data_shape"] = data.shape
                report["data_finite_fraction"] = finite_fraction(data)
                report["data_abs_sum_first_particles_preview"] = preview_nested(abs_sum.tolist())
                report["mask_source"] = f"data[..., {MASK_FEATURE_INDEX}] != 0"
                report["particle_counts_from_mask"] = particle_mask.sum(axis=1).astype(int).tolist()
                report["zero_row_mask_disagreements"] = np.logical_xor(abs_sum > 0, particle_mask).sum(axis=1).astype(int).tolist()
            if "jet" in handle:
                jet = np.asarray(handle["jet"][:n])
                report["jet_shape"] = jet.shape
                report["jet_finite_fraction"] = finite_fraction(jet)
            if "pid" in handle:
                pid = np.asarray(handle["pid"][:n])
                report["pid_shape"] = pid.shape
                report["pid_preview"] = preview_nested(pid.tolist())
            return report
    except Exception as exc:
        return {"path": str(path), "error": f"{type(exc).__name__}: {exc}"}


if hdf5_files:
    tiny_report = inspect_tiny_slice(hdf5_files[0], n=8)
    print(json.dumps(tiny_report, indent=2))
else:
    print("No HDF5 file available for slice inspection.")

## 6. Feature Configuration Practice: A-D Mapping

The formal project uses nested feature configurations:

| Config | Intended particle-level input | Purpose |
|---|---|---|
| A | Kinematics only | Baseline value of four-momentum information |
| B | A + charge | Incremental value of electric charge |
| C | B + PID | Incremental value of particle identity |
| D | C + impact-parameter / vertex-quality candidates | Strongest allowed detector-level representation |

Do not freeze the column map from memory. Fill `COLUMN_MAP` only after inspecting real JetClass files and preprocessing scripts.

In [ ]:
# Load the semantic ladder from one project-wide source. Real columns remain unresolved until E0.
with FEATURE_LADDER_PATH.open("r", encoding="utf-8") as handle:
    FEATURE_LADDER = json.load(handle)
FEATURE_CONFIG_GROUPS: dict[str, list[str]] = FEATURE_LADDER["config_groups"]

# Fill every group with documented columns from the selected data source after E0 inspection.
# PID should expand to one-hot columns by default; do not encode categories as one ordinal scalar.
COLUMN_GROUPS: dict[str, list[str]] = {group: [] for groups in FEATURE_CONFIG_GROUPS.values() for group in groups}
COLUMN_MAP: dict[str, int] = {}


def resolve_feature_names(config_name: str, column_groups: dict[str, list[str]]) -> list[str]:
    if config_name not in FEATURE_CONFIG_GROUPS:
        raise ValueError(f"Unknown config: {config_name}")
    empty_groups = [group for group in FEATURE_CONFIG_GROUPS[config_name] if not column_groups.get(group)]
    if empty_groups:
        raise KeyError(f"Unresolved column groups for config {config_name}: {empty_groups}")
    feature_names = [name for group in FEATURE_CONFIG_GROUPS[config_name] for name in column_groups[group]]
    if len(feature_names) != len(set(feature_names)):
        raise ValueError(f"Duplicate feature names for config {config_name}: {feature_names}")
    return feature_names


def validate_feature_config(column_map: dict[str, int], feature_names: Sequence[str], total_features: int | None = None) -> list[int]:
    missing = [name for name in feature_names if name not in column_map]
    if missing:
        raise KeyError(f"Missing columns in COLUMN_MAP: {missing}")
    indices = [column_map[name] for name in feature_names]
    if len(indices) != len(set(indices)):
        raise ValueError(f"Duplicate column indices in {feature_names}: {indices}")
    if total_features is not None:
        out_of_bounds = [idx for idx in indices if idx < 0 or idx >= total_features]
        if out_of_bounds:
            raise IndexError(f"Column indices outside [0, {total_features}): {out_of_bounds}")
    return indices


def slice_feature_config(
    data: np.ndarray,
    config_name: str,
    column_map: dict[str, int],
    column_groups: dict[str, list[str]],
) -> np.ndarray:
    feature_names = resolve_feature_names(config_name, column_groups)
    indices = validate_feature_config(column_map, feature_names, data.shape[-1])
    return data[..., indices]


if hdf5_files and COLUMN_MAP and all(COLUMN_GROUPS.values()):
    try:
        with h5py.File(hdf5_files[0], "r") as handle:
            sample_data = np.asarray(handle["data"][:4])
        for config_name in FEATURE_CONFIG_GROUPS:
            features = slice_feature_config(sample_data, config_name, COLUMN_MAP, COLUMN_GROUPS)
            print(config_name, features.shape, "finite_fraction=", finite_fraction(features))
    except Exception as exc:
        print(f"Feature-config slice skipped: {type(exc).__name__}: {exc}")
else:
    print("COLUMN_GROUPS/COLUMN_MAP are unresolved. Fill them after inspecting real feature columns; this is expected during first setup.")

## 6.1 Helper Sanity Tests

These tests use tiny synthetic data and run quickly. They do not validate physics or JetClass provenance; they only check that the notebook helpers fail in predictable ways.

In [ ]:
if h5py is None:
    print("h5py is not installed; skipping helper sanity tests.")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        valid_path = tmpdir_path / "valid.h5"
        missing_path = tmpdir_path / "missing.h5"

        with h5py.File(valid_path, "w") as handle:
            handle.create_dataset("data", data=np.zeros((2, 3, 10), dtype=np.float32))
            handle.create_dataset("jet", data=np.zeros((2, 4), dtype=np.float32))
            handle.create_dataset("pid", data=np.zeros((2, 2), dtype=np.float32))

        with h5py.File(missing_path, "w") as handle:
            handle.create_dataset("other", data=np.zeros((2,), dtype=np.float32))

        valid_result = check_omnilearn_input_contract(valid_path)
        missing_result = check_omnilearn_input_contract(missing_path)
        assert not valid_result.issues, valid_result
        assert "missing 'data' dataset" in missing_result.issues, missing_result
        assert validate_feature_config({"a": 0, "b": 1}, ["a", "b"], total_features=2) == [0, 1]

        try:
            validate_feature_config({"a": 0}, ["a", "b"], total_features=2)
        except KeyError:
            pass
        else:
            raise AssertionError("Missing feature names should raise KeyError")

        try:
            validate_feature_config({"a": 2}, ["a"], total_features=2)
        except IndexError:
            pass
        else:
            raise AssertionError("Out-of-bounds feature index should raise IndexError")

    print("Helper sanity tests passed.")

## 7. Pretrained Checkpoint Inventory

Before E0.5, every checkpoint artifact needs source, license, exact path, and SHA-256 hash. This cell only records local files; it does not prove compatibility with the model graph.

Hashing very large checkpoint files can take time. Keep `MAX_HASH_BYTES` conservative during practice and raise it intentionally for formal E0.5 provenance capture.

In [ ]:
CHECKPOINT_SUFFIXES = {".h5", ".keras", ".ckpt", ".index", ".pt", ".pth", ".npz"}
TF_SHARD_RE = re.compile(r"\.data-\d{5}-of-\d{5}$")
MAX_HASH_BYTES = 2 * 1024**3


def looks_like_checkpoint(path: Path) -> bool:
    name = path.name.lower()
    if any(name.endswith(suffix) for suffix in CHECKPOINT_SUFFIXES):
        return True
    if TF_SHARD_RE.search(name):
        return True
    return name == "checkpoint"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def checkpoint_artifact_key(path: Path) -> Path | None:
    name = path.name
    if name.lower() == "checkpoint":
        return None
    if name.lower().endswith(".index"):
        return path.with_name(name[:-len(".index")])
    shard_match = TF_SHARD_RE.search(name.lower())
    if shard_match:
        return path.with_name(name[:shard_match.start()])
    return path


def group_checkpoint_artifacts(paths: Iterable[Path]) -> list[tuple[Path, tuple[Path, ...]]]:
    grouped: dict[Path, list[Path]] = {}
    for path in sorted(set(paths)):
        artifact_key = checkpoint_artifact_key(path)
        if artifact_key is not None:
            grouped.setdefault(artifact_key, []).append(path)
    return [(key, tuple(sorted(members))) for key, members in sorted(grouped.items())]


def sha256_artifact(paths: Sequence[Path]) -> str:
    digest = hashlib.sha256()
    for path in sorted(paths):
        digest.update(path.name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(bytes.fromhex(sha256_file(path)))
    return digest.hexdigest()


def discover_checkpoint_files(candidate_dirs: Iterable[Path]) -> list[Path]:
    paths: list[Path] = []
    for directory in candidate_dirs:
        if directory.exists():
            paths.extend(path for path in directory.rglob("*") if path.is_file() and looks_like_checkpoint(path))
    return sorted(set(paths))


checkpoint_component_files = discover_checkpoint_files(CHECKPOINT_CANDIDATE_DIRS)
checkpoint_artifacts = group_checkpoint_artifacts(checkpoint_component_files)
checkpoint_files = [load_path for load_path, _ in checkpoint_artifacts]
print(f"Found {len(checkpoint_artifacts)} checkpoint artifact(s) from {len(checkpoint_component_files)} component file(s).")

checkpoint_report = []
for load_path, component_files in checkpoint_artifacts[:20]:
    size_bytes = sum(path.stat().st_size for path in component_files)
    record = {
        "path": str(load_path),
        "component_files": [str(path) for path in component_files],
        "size_bytes": size_bytes,
        "sha256": None,
        "hash_skipped": False,
    }
    if size_bytes > MAX_HASH_BYTES:
        record["hash_skipped"] = True
        record["skip_reason"] = f"artifact exceeds MAX_HASH_BYTES={MAX_HASH_BYTES}"
    else:
        record["sha256"] = sha256_artifact(component_files)
    checkpoint_report.append(record)
checkpoint_report_truncated = len(checkpoint_artifacts) > 20

print(json.dumps(checkpoint_report, indent=2))

if not checkpoint_artifacts:
    print("\nNo checkpoint files found. For E0.5, place official pretrained checkpoints under one of:")
    for directory in CHECKPOINT_CANDIDATE_DIRS:
        print("-", directory)

## 8. Official Command Templates

These are dry-run command templates based on the official OmniLearn README. Do not run large preprocessing or training from inside this notebook unless you intentionally set up the data, checkpoint, and GPU environment.

The commands are shown in the official bash-style shape used by the reference repository. On Windows/PowerShell or Google Colab, adapt path quoting and directory changes before execution.

In [ ]:
def print_command_block(title: str, commands: Sequence[str]) -> None:
    print(f"\n# {title}")
    for command in commands:
        print(command)


folder = str(DATA_DIR)
reference = str(REFERENCE_DIR)

print_command_block("Clone official OmniLearn reference", [
    f"git clone {OMNILEARN_REPO_URL}.git {reference}",
])

print_command_block("Preprocess JetClass files with official OmniLearn script", [
    f"cd {reference}/preprocessing",
    f"python preprocess_jetclass.py --sample train --folder {folder}",
    f"python preprocess_jetclass.py --sample val --folder {folder}",
    f"python preprocess_jetclass.py --sample test --folder {folder}",
])

print_command_block("Train OmniLearn from scratch on JetClass, official full-scale command", [
    f"cd {reference}/scripts",
    "python train.py --dataset jetclass --lr 3e-5 --layer_scale --local --mode all",
])

print_command_block("Fine-tune a pretrained classifier on an official supported classification dataset", [
    f"cd {reference}/scripts",
    "python train.py --dataset top --layer_scale --local --mode classifier --warm_epoch 3 --epoch 10 --stop_epoch 3 --batch 256 --wd 0.1 --fine_tune",
    "python evaluate_classifiers.py --batch 1000 --local --layer_scale --dataset top --fine_tune",
])

print("\nFor this project's JetClass top-vs-QCD feature-availability study, the formal code should add or adapt a dataloader so that dataset selection, A-D feature views, split manifests, and run records are controlled by this repository.")
print(f"OmniLearn paper: {OMNILEARN_PAPER}")
print(f"JetClass source of truth: {JETCLASS_ZENODO}")

## 9. Optional Tiny Masked-Set Smoke Test

This is not OmniLearn. It is a tiny PyTorch masked-set classifier used only to confirm that the local environment can perform a forward and backward pass on particle-set tensors. It helps separate generic environment problems from OmniLearn-specific integration problems.

Skip this cell if PyTorch is not installed.

In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    nn = None


if torch is None:
    print("PyTorch is not installed; skipping tiny smoke test.")
else:
    torch.manual_seed(123)

    class TinyMaskedSetClassifier(nn.Module):
        def __init__(self, input_dim: int, hidden_dim: int = 32):
            super().__init__()
            self.particle_net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
            )
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
            encoded = self.particle_net(x)
            mask_f = mask.unsqueeze(-1).to(encoded.dtype)
            pooled = (encoded * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
            return self.classifier(pooled).squeeze(-1)

    batch_size, n_particles, n_features = 8, 16, 4
    x = torch.randn(batch_size, n_particles, n_features)
    mask = torch.rand(batch_size, n_particles) > 0.25
    y = torch.randint(0, 2, (batch_size,), dtype=torch.float32)

    model = TinyMaskedSetClassifier(input_dim=n_features)
    logits = model(x, mask)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
    loss.backward()

    finite_grads = all(
        parameter.grad is not None and torch.isfinite(parameter.grad).all().item()
        for parameter in model.parameters()
    )
    assert logits.shape == (batch_size,)
    assert torch.isfinite(loss).item()
    assert finite_grads
    print({
        "logits_shape": tuple(logits.shape),
        "loss": float(loss.detach()),
        "finite_gradients": bool(finite_grads),
    })

## 10. E0.5 Adapter-Loading Spike Checklist

Use this checklist before starting formal E1 pilot runs.

| Check | Evidence to save |
|---|---|
| Checkpoint artifact | Source URL, license, local path, SHA-256 |
| Input schema | `data`, `jet`, `pid` shapes, feature dictionary, normalization source |
| Adapter policy | `adapter_swap` preferred, `fixed_dim_neutralization` only as fallback |
| Non-input weight loading | Loaded layer names and skipped layer names with reasons |
| A/B/C/D forward pass | Finite logits for each feature configuration on the dev slice |
| Gradient flow | Finite gradients in trainable layers after one backward pass |
| Tiny fine-tune | One tiny epoch reduces training loss relative to the first batch |
| Fallback decision | If checkpoint loading fails, downgrade claims to supervised PET-style feature ablation |

In [ ]:
_current_checkpoint_files = globals().get("checkpoint_files", [])
_current_checkpoint_report = globals().get("checkpoint_report", [])
_current_env_report = globals().get("env_report", {"error": "Environment inventory cell has not been run"})
_current_reference_report = globals().get("reference_report", {"error": "Reference checkout cell has not been run"})

run_record_template = {
    "run_id": "practice_pretrained_omnilearn_pet_001",
    "stage": "pre_E0_5_practice",
    "git_commit": safe_git_value(["rev-parse", "HEAD"], cwd=PROJECT_ROOT),
    "data_manifest_hash": None,
    "dataset_hash": None,
    "preprocessing_version": None,
    "pretraining_source": OMNILEARN_REPO_URL,
    "reference_git_commit": _current_reference_report.get("git_commit"),
    "reference_git_describe": _current_reference_report.get("git_describe"),
    "checkpoint_path": str(_current_checkpoint_files[0]) if _current_checkpoint_files else None,
    "checkpoint_sha256": _current_checkpoint_report[0].get("sha256") if _current_checkpoint_report else None,
    "checkpoint_component_files": _current_checkpoint_report[0].get("component_files", []) if _current_checkpoint_report else [],
    "feature_config": None,
    "model_name": "OmniLearn/PET-style backbone",
    "initialization": "pretrained",
    "seed": 123,
    "training_size_per_class": None,
    "hyperparameters": {},
    "input_adapter_policy": "adapter_swap",
    "loaded_layers": [],
    "skipped_layers": [],
    "hardware_spec": _current_env_report,
    "runtime_seconds": None,
    "peak_gpu_memory_mb": None,
    "best_validation_epoch": None,
    "test_metrics": None,
    "prediction_file": None,
    "failure_status": "practice_not_a_formal_run",
}

print(json.dumps(run_record_template, indent=2))

## Passing Criteria for This Practice Notebook

The practice notebook has done its job when:

1. the minimum Python practice packages are available;
2. the official OmniLearn repository can be cloned or the network failure is recorded;
3. at least one small JetClass-like HDF5 file can be inspected, or the absence of data is explicit;
4. candidate feature columns for A-D are not guessed from memory;
5. checkpoint files, if present, have SHA-256 hashes recorded;
6. command templates for preprocessing, fine-tuning, and evaluation are visible;
7. any formal training remains outside this notebook until E0 and E0.5 gates are passed.

This notebook is not E0 evidence until it has been executed against a real JetClass dev slice and the resulting environment, data-schema, checkpoint, and failure-status outputs have been reviewed.

If any criterion fails, record the failure in `docs/preparation/reproduction-log.md`, keep the project before E0/E0.5, and resolve the missing data, dependency, checkpoint, or schema item before launching pilot training.

Next project artifact after this notebook: a reproduction log under `docs/preparation/reproduction-log.md` recording the exact OmniLearn checkout, environment, data paths, checkpoint paths, and first successful or failed reference run.